In [68]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
import scipy.stats as stats
from mpl_toolkits.mplot3d import Axes3D

#### Question 1
##### a. Omitted variable bias: the regressor X is correlated with an omitted variable that determines the outcome of the variable Y
##### b. Partial effect: the impact that one parameter has on Y while keeping all other parameters constant
##### c. The difference between $R^2$ and adjusted $R^2$: $R^2$ is how much of the variation is explained by the regressor(s) while adjusted $R^2$ penalizes regressors if they aren't relevant
##### d. The dummy variable trap: the perfect multicollinearity among the dummy variables

#### Question 2 
In this exercise, we will work with the data file birthweight_smoking.csv , which contains data for a random sample of babies born in Pennsylvania in 1989. The data includes the baby’s birth weight together with various characteristic of the mother, including whether she smoked during the pregnancy.
##### a) Run a regression between birthweight and smoker . Compute heteroskedasticity robust standard errors. Interpret the estimated coefficient on smoker. 
##### b) Regress birthweight on smoker , alcohol , nprevist and educ . Interpret the estimated coefficient on smoker . Compare your result with part (a). 
##### c) Consider regression in part (b). Interpret the estimated coefficients on nprevist and educ . 
##### d) Consider the regression in part (b). Interpret $R^2$ and adjusted $R^2$. Why they are so similar? 
##### e) Jane smoked during her pregnancy, did not drink alcohol, had 8 prenatal care visits and has 12 years of educational attainment. Use the regression in part (b) to predict the birth weight of Jane’s child. 
##### f) An alternative way to control for prenatal visits is to use the binary variables tripre0 through tripre3 . Regress birthweight on smoker , alcohol , tripre1 , tripre2 , tripre3 and tripre0 . Note that R will not estimate the coefficient associated with tripre0 in order to avoid the dummy variable trap. Show your result and interpret the estimated coefficient on tripre1 .

In [69]:
BWsmoking = pd.read_csv('birthweight_smoking.csv', header = 0)
BWsmoking.columns

Index(['nprevist', 'alcohol', 'tripre1', 'tripre2', 'tripre3', 'tripre0',
       'birthweight', 'smoker', 'unmarried', 'educ', 'age', 'drinks'],
      dtype='object')

In [70]:
X = sm.add_constant(BWsmoking['smoker'])
y = BWsmoking['birthweight']

In [71]:
model1 = smf.ols(formula='birthweight ~ smoker', data=BWsmoking)
result1 = model1.fit()
print(result1.summary())

                            OLS Regression Results                            
Dep. Variable:            birthweight   R-squared:                       0.029
Model:                            OLS   Adj. R-squared:                  0.028
Method:                 Least Squares   F-statistic:                     88.28
Date:                Sun, 26 Apr 2026   Prob (F-statistic):           1.09e-20
Time:                        20:50:12   Log-Likelihood:                -23364.
No. Observations:                3000   AIC:                         4.673e+04
Df Residuals:                    2998   BIC:                         4.674e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   3432.0600     11.871    289.115      0.0

##### a) The coefficient on smoker is about -253 so that means that if the mother smokes during pregnancy, it will result in the birthweight decreasing by about 253 grams.

In [72]:
model2 = smf.ols( formula =' birthweight ~ smoker + alcohol + nprevist + educ', data = BWsmoking)
result2 = model2.fit(cov_type = 'HC1')
print(result2.summary())

                            OLS Regression Results                            
Dep. Variable:            birthweight   R-squared:                       0.074
Model:                            OLS   Adj. R-squared:                  0.072
Method:                 Least Squares   F-statistic:                     45.58
Date:                Sun, 26 Apr 2026   Prob (F-statistic):           3.23e-37
Time:                        20:50:24   Log-Likelihood:                -23293.
No. Observations:                3000   AIC:                         4.660e+04
Df Residuals:                    2995   BIC:                         4.663e+04
Df Model:                           4                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   2954.6037     74.786     39.507      0.0

In [73]:
models = ['Model 1', 'Model 2']
results_table = summary_col(results=[result1, result2],
float_format='%0.3f',
stars=True,
model_names=models)
results_table

,Model 1,Model 2
Intercept,3432.060***,2954.604***
,(11.871),(74.786)
smoker,-253.228***,-207.932***
,(26.951),(27.115)
alcohol,,-35.491
,,(72.759)
nprevist,,33.168***
,,(3.642)
educ,,8.118
,,(5.012)


##### b) The estimated coeficient on smoker is about -208, holding alcohol, nprevisit, and educ constant. Compared to the coefficient computed in part (a), it became less negative after adding controls, suggesting that in part (a) regressor smoker was picking up some of the effect of the other harmful behaviors, i.e. alcohol, which caused downward omitted variable bias.
##### c) The estimated coefficient on nprevisit is about 33, which suggests that while holding smoker, alcohol, and educ constant, each additional prenatal visit increased the overall birth weight by about 33 grams. The estimated coefficient on educ is about 8, which suggests that while holding smoker, alcohol, and nprevisit constant, each additional years of education of the mother increased the birth weight by about 8 grams. 
##### d) The $R^2$ and adjusted $R^2$ are so similar in the second regression because the regressors added in the second model are relevant. Adjusted $R^2$ only penalizes regressors if they are not relevant to the model and is only slightly smaller when the regressors are relevant. As a result, $R^2$ and adjusted $R^2$ are nearly identical.

##### e) birthweight(Jane)  = 2954.604 -207.932(1) - 35.491(0) + 33.168(8) + 8.118(12) = 3109.432
##### The birthweight of Jane's child is approximately 3109.432 grams.

In [74]:
model3 = smf.ols( formula =' birthweight ~ smoker + alcohol + tripre1 + tripre2 + tripre3', data = BWsmoking)
result3 = model3.fit(cov_type = 'HC1')
print(result3.summary())

                            OLS Regression Results                            
Dep. Variable:            birthweight   R-squared:                       0.046
Model:                            OLS   Adj. R-squared:                  0.045
Method:                 Least Squares   F-statistic:                     23.22
Date:                Sun, 26 Apr 2026   Prob (F-statistic):           5.99e-23
Time:                        20:50:32   Log-Likelihood:                -23336.
No. Observations:                3000   AIC:                         4.668e+04
Df Residuals:                    2994   BIC:                         4.672e+04
Df Model:                           5                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   2756.5806    146.077     18.871      0.0

##### f) The coefficient of tripre1 is about 698, meaning while holding smoker, alcohol, tripre2, and tripre3 constant, the birthweight increases by about 698 grams if the mother had her first prenatal care visit in her first trimester.

#### Question 3
Each month the Bureau of Labor Statistics in the U.S. Department of Labor conducts the Current Population Survey (CPS), which provides data on labor force characteristics of the population, including the level of employment, unemployment, and earnings. Approximately 65,000 randomly selected U.S. households are surveyed each month. The sample is chosen by randomly selecting addresses from a database comprised of adddresses from the most recent decennial census augmented with data on new housing units constructed after the last census. The exact randome sampling scheme is rather complicated (first small geographical areas are selected, then housing units within these areas randomly selected); details can be found in the Handbook of Labor Statistics and is described on the Bureau of Labor Statistics website (www.bls.gov). For this exercise, you will utilize the data set called cpsDATA.csv.
##### a) Run a regression of (average hourly) earnings ahe on education yrseduc, female and age. Compute heteroskedasticity robust standard errors. What is the estimated coeefficient on yrseduc? 
##### b) Is the estimated education effect significantly different than zero at the 5% significance level?
##### c) To the regression in part (a), add the region dummies midwest, south, and west. Do not include northeast to avoid the dummy variable trap. Interpret the estimated coefficient on midwest. 
##### d) Juanita is a 30 year old female from the South with 12 years of education. Jennifer is a 25 year old female from the West with 16 years of education. What is the predeicted difference in their hourly earnings?

In [75]:
CPSData = pd.read_csv('cpsDATA.csv')

In [76]:
print(CPSData[['ahe', 'yrseduc', 'female', 'age']].describe().round(3))

             ahe    yrseduc     female        age
count  59668.000  59668.000  59668.000  59668.000
mean      25.214     14.094      0.439     42.885
std       17.289      2.598      0.496     12.584
min        2.000      6.000      0.000     15.000
25%       13.736     12.000      0.000     33.000
50%       20.192     13.000      0.000     42.000
75%       31.250     16.000      1.000     52.000
max      228.365     20.000      1.000     85.000


In [77]:
model1 = smf.ols(formula='ahe ~ yrseduc + female + age', data=CPSData)
result1 = model1.fit(cov_type = 'HC1')
print(result1.summary())

                            OLS Regression Results                            
Dep. Variable:                    ahe   R-squared:                       0.250
Model:                            OLS   Adj. R-squared:                  0.250
Method:                 Least Squares   F-statistic:                     4601.
Date:                Sun, 26 Apr 2026   Prob (F-statistic):               0.00
Time:                        20:53:06   Log-Likelihood:            -2.4615e+05
No. Observations:               59668   AIC:                         4.923e+05
Df Residuals:                   59664   BIC:                         4.923e+05
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -23.1342      0.419    -55.259      0.0

##### a) The estimated coefficient of yrseduc is about 3 keeping female and age constant, meaning every additional year of education the average hourly earnings increased by about 3 dollars.

In [78]:
models = ['Model 1']
results_table = summary_col(results=[result1],
float_format='%0.3f',
stars=True,
model_names=models)
results_table

,Model 1
Intercept,-23.134***
,(0.419)
yrseduc,2.948***
,(0.028)
female,-6.410***
,(0.122)
age,0.224***
,(0.005)
R-squared,0.250
R-squared Adj.,0.250


##### b) Yes, the estimated education effect is significantly different than zero at the 5% significance level. This is because with the table above, it is shown that it is significant at the 1% level so it is also significant at the 5% level.

In [79]:
model2 = smf.ols(formula='ahe ~ yrseduc + female + age + midwest + south + west', data=CPSData)
result2 = model2.fit(cov_type = 'HC1')
print(result2.summary())

                            OLS Regression Results                            
Dep. Variable:                    ahe   R-squared:                       0.252
Model:                            OLS   Adj. R-squared:                  0.252
Method:                 Least Squares   F-statistic:                     2322.
Date:                Sun, 26 Apr 2026   Prob (F-statistic):               0.00
Time:                        20:54:57   Log-Likelihood:            -2.4607e+05
No. Observations:               59668   AIC:                         4.921e+05
Df Residuals:                   59661   BIC:                         4.922e+05
Df Model:                           6                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -21.6241      0.454    -47.616      0.0

##### c) The estimated coefficient of midwest is about -2, meaning while holding yrseduc, female, age, south, and west constant, the average hourly earning decereases by about 2 dollars.

##### d) ahe(Juanita) = -21.6241 + 2.9387(12) - 6.3904(1) + 0.2225(30) - 1.8961(1) = 12.0288
##### ahe(Jennifer) = -21.6241 + 2.9387(16) - 6.3904(1) + 0.2225(25) - 0.7109(1) = 23.8563
##### 23.8563 - 12.0288 = 11.8275
##### The predicted difference in hourly earnings is that Jennifer will be making about 11.83 dollars more than Jaunita. 